In [9]:
from openai import OpenAI

def invoke_llm(system_prompt: str, user_prompt: str):
    """Calls the LLM without history.
    LLM function call to be passed to strictjson.strict_json(). It is advised not to change the parameters.
    Refer to https://github.com/tanchongmin/strictjson/blob/main/strictjson/base.py#L319

    Args:
        system_prompt (str): System prompt.
        user_prompt (str): User prompt.

    Returns:
        str: LLM response string.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]   

    client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

    response = client.chat.completions.create(
        model="lmstudio-community/Meta-Llama-3.1-8B-Instruct-GGUF",
        messages=messages,
        temperature=1,
    )

    return response.choices[0].message.content

In [10]:
ER_SYSTEM_PROMPT = """
You are an expert in the gastronomy domain with extensive knowledge of various pizzas. Your task is to 
accurately identify and extract the names of entities from the given text. You should focus on recognizing 
specific terminology and contextually relevant phrases that pertain to objects such as food and beverages 
within the gastronomy sector. Your output should be in JSON format, categorizing the identified terms into 
"entities". The output should be strictly the JSON object without any additional commentary or explanation.
"""

ER_USER_PROMPT = """
I want you to identify the entities of the following input.

Input: {input}
"""

### Testing strictjson

In [11]:
from strictjson import strict_json

def entity_recognition(input: str) -> dict:
    """
    Performs entity recognition on the provided input string using a large language model (LLM).

    This function identifies and categorizes entities within the user input.

    Multiple 

    Args:
        input (str): The input string to be analyzed by the LLM.

    Returns:
        dict: A dictionary with key 'entities', containing the identified entities within the question.

    Example:
        Input: "Can Creality Ender manufacture flange?"
        Output: {'entities': ['Creality Ender', 'flange']}

    """
    response = strict_json(
        system_prompt=ER_SYSTEM_PROMPT,
        user_prompt=ER_USER_PROMPT.format(input=input),
        output_format={"entities": "Array of entities"},
        llm=invoke_llm
    )

    return response

In [12]:
question = "Can EOSM manufacture a flange?"
result = entity_recognition(question)

In [13]:
result

{'entities': ['EOSM', 'flange']}

In [3]:
import chromadb

persistent_client = chromadb.PersistentClient(path='../../manon_chat_interface/data/vectorstores/entities')
machine_collection = persistent_client.get_or_create_collection("machines_collection")
part_collection = persistent_client.get_or_create_collection("parts_collection")

In [4]:
machine_collection.get()

{'ids': ['00565f46-184d-5907-a3cf-23a0f539e985',
  '0490201f-bda0-500f-be61-32e7ab2683df',
  '0dcc5b40-4e62-5763-b267-2edf67084532',
  'a4005aa8-7006-5ca0-95ae-b50a578f02c8',
  'c34dd14c-f9fb-5be6-a039-63aa00f8f702',
  'd22d2be6-a29e-5dd0-a83a-7616492f9318',
  'fc0107db-bdb4-50a3-8a0b-5d8e14750efa'],
 'embeddings': None,
 'metadatas': [{'IRI': 'http://www.ontology.ift.dlr.de/MANON/CrealityEnder3Pro#M_CrealityEnder3Pro'},
  {'IRI': 'http://www.ontology.ift.dlr.de/MANON/UltimakerS5#M_UltimakerS5'},
  {'IRI': 'http://www.ontology.ift.dlr.de/MANON/DremelDigiLab3D45#M_DremelDigiLab3D45'},
  {'IRI': 'http://www.ontology.ift.dlr.de/MANON/EOSM290#M_EOSM290'},
  {'IRI': 'http://www.ontology.ift.dlr.de/MANON/EOSP396#M_EOSP396'},
  {'IRI': 'http://www.ontology.ift.dlr.de/MANON/EOSM290Test#M_EOSM290Test'},
  {'IRI': 'http://www.ontology.ift.dlr.de/MANON/EWMTitanXQ500PulsDW#M_EWMTitanXQ500PulsDW'}],
 'documents': ['CrealityEnder3Pro',
  'UltimakerS5',
  'DremelDigiLab3D45',
  'EOSM290',
  'EOSP396'

In [26]:
machine_collection.query(
    query_texts=res['machines'],
    n_results=1
)

{'ids': [['00565f46-184d-5907-a3cf-23a0f539e985'],
  ['0dcc5b40-4e62-5763-b267-2edf67084532']],
 'distances': [[0.6950375067210832], [0.668053136108828]],
 'metadatas': [[{'IRI': 'http://www.ontology.ift.dlr.de/MANON/EOSM290#M_EOSM290'}],
  [{'IRI': 'http://www.ontology.ift.dlr.de/MANON/CrealityEnder3Pro#M_CrealityEnder3Pro'}]],
 'embeddings': None,
 'documents': [['EOSM290'], ['CrealityEnder3Pro']],
 'uris': None,
 'data': None}

In [2]:
import json

data = {'sparql_query': "PREFIX pizza: <http://www.co-ode.org/ontologies/pizza#> \n    PREFIX owl: <http://www.w3.org/2002/07/owl#>\n    SELECT ?p ?o\n    WHERE {\n      ?x rdfs:label 'American'.\n      ?y rdfs:subClassOf ?x.\n      ?z rdf:type pizza:PizzaBase.\n      OPTIONAL { ?y owl:onProperty pizza:hasIngredient. }\n      FILTER NOT EXISTS {?y rdfs:comment 'unclosed'. }\n      ?t1 owl:sameAs <http://www.co-ode.org/ontologies/pizza#MozzarellaTopping>.\n      ?p a pizza:Pizza.\n      ?p rdfs:subClassOf ?y.\n      ?o rdf:type ?t1.\n    }", 'explanation': "This query asks if an American-style pizza can have a mozzarella topping. It first defines the class of American pizzas by finding all subclasses of 'American', and then checks each subclass to see if it has any toppings. Finally, it checks if there are any instances that match both these conditions."}

sparql_query = data['sparql_query']
explanation = data['explanation']

In [4]:
print(sparql_query)

PREFIX pizza: <http://www.co-ode.org/ontologies/pizza#> 
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    SELECT ?p ?o
    WHERE {
      ?x rdfs:label 'American'.
      ?y rdfs:subClassOf ?x.
      ?z rdf:type pizza:PizzaBase.
      OPTIONAL { ?y owl:onProperty pizza:hasIngredient. }
      FILTER NOT EXISTS {?y rdfs:comment 'unclosed'. }
      ?t1 owl:sameAs <http://www.co-ode.org/ontologies/pizza#MozzarellaTopping>.
      ?p a pizza:Pizza.
      ?p rdfs:subClassOf ?y.
      ?o rdf:type ?t1.
    }
